# Hyperparameter grid search
This final section performs a hyperparameter grid search for the Rank-LSTM and ReRa-LSTM model. The performance of a neural network is often highly sensitive to its architectural and training parameters. A grid search is a systematic method for finding the optimal combination of these parameters.

The code defines a grid of potential values for key hyperparameters. It then iterates through every possible combination of these values, training and evaluating a new model for each. The performance of each combination is recorded, and the combination that yields the best performance (in this case, measured by the backtesting return btl) on the test set is identified as the optimal configuration. 

In [5]:
import argparse
import copy
import numpy as np
import os
import sys
import random
import torch
import pandas as pd
import itertools
from time import time

sys.path.append(os.path.abspath('../../'))
from models.lstm import LSTMModel, LSTM
from models.rank_lstm import RankLSTMModel, RankLSTM
from models.rel_rank_lstm import ReRaLSTMModel, ReRaLSTM
from models.evaluate import evaluate
from models.data_loading import load_EOD_data, load_relation_data

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Hyperparameter Search for Rank-LSTM

In [6]:
parameters = {'seq': 16, 'unit': 64, 'lr': 0.001, 'alpha': 0.1}
data_path='../../data/2013-01-01'
market_name='NASDAQ'
tickers_fname='NASDAQ_tickers_qualify_dr-0.98_min-5_smooth.csv'
relation_name='wikidata'
emb_fname='NASDAQ_rank_lstm_seq-16_unit-64_2.csv.npy'

In [ ]:
import pandas as pd
import itertools

print("="*60)
print("Starting Hyperparameter Grid Search for RankLSTM")
print("="*60)

# Define the grid of hyperparameters to search over
param_grid = {
    'seq': [2, 4, 8, 16],
    'unit': [16, 32, 64, 128],
    'alpha': [0.1, 1.0, 10.0]
}

# Base parameters
base_parameters = {
    'lr': 0.001
}

# Store the results of each run
results = []
best_irr = -float('inf')
best_params = {}
best_perf = {}

# Generate all combinations of hyperparameters
keys, values = zip(*param_grid.items())
hyperparam_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

total_combinations = len(hyperparam_combinations)
print(f"Total combinations to test: {total_combinations}\n")


for i, params in enumerate(hyperparam_combinations):
    current_params = base_parameters.copy()
    current_params.update(params)
    
    print(f"--- Testing Combination {i+1}/{total_combinations}: {current_params} ---")

    # 1. Instantiate RankLSTM with the current set of hyperparameters
    temp_rank_lstm = RankLSTM(
        data_path=data_path,
        market_name=market_name,
        tickers_fname=tickers_fname,
        parameters=current_params,
        steps=1,
        epochs=50,  # Using a consistent number of epochs
        batch_size=None,
        gpu=torch.cuda.is_available()
    )

    # 2. Train the model and get the performance of the best validation epoch
    _, _, _, test_pred, test_gt, test_mask = temp_rank_lstm.train()
    
    # 3. Evaluate the performance, focusing on IRR (btl)
    current_perf = evaluate(test_pred, test_gt, test_mask)
    current_irr = current_perf.get('btl', -float('inf'))
    
    print(f"Combination {i+1} Result: IRR (btl) = {current_irr:.4f}, MSE = {current_perf['mse']:.6f}, MRR = {current_perf['mrrt']:.4f}")
    
    # 4. Store and track the best results
    results.append({**current_params, **current_perf})
    
    if current_irr > best_irr:
        best_irr = current_irr
        best_params = current_params
        best_perf = current_perf
        print(f"\n*** New Best IRR Found: {best_irr:.4f} ***\n")

# --- Display Final Results ---
print("\n" + "="*60)
print("Grid Search Complete!")
print("="*60)

results_df = pd.DataFrame(results)
print("Full Grid Search Results:")
print(results_df[['seq', 'unit', 'alpha', 'btl', 'mse', 'mrrt']].sort_values(by='btl', ascending=False).to_string())

print("\n" + "="*60)
print("Best Hyperparameters Found")
print("="*60)
print(f"Parameters: {best_params}")
print(f"Achieved IRR (btl): {best_perf.get('btl', 'N/A'):.4f}")
print(f"Achieved MSE: {best_perf.get('mse', 'N/A'):.6f}")
print(f"Achieved MRR: {best_perf.get('mrrt', 'N/A'):.4f}")

## Hyperparameter Search for ReRa-LSTM

In [ ]:
print("="*60)
print(f"Starting Hyperparameter Grid Search for ReRaLSTM on {market_name}")
print(f"Using Relation: {relation_name}")
print("="*60)

# Define the grid of hyperparameters to search over
param_grid = {
    'alpha': [0.1, 1.0, 10.0],
    'in_pro': [True, False], # True=Explicit(RSR_E), False=Implicit(RSR_I)
    'flat': [False] # Architectural choice, fixed for this search
}

# Base parameters - 'unit' must match the dimension of the loaded embeddings
base_parameters = {
    'lr': 0.001,
    'unit': 32
}

# Store the results of each run
results = []
best_irr = -float('inf')
best_params = {}
best_perf = {}

# Generate all combinations of hyperparameters
keys, values = zip(*param_grid.items())
hyperparam_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

total_combinations = len(hyperparam_combinations)
print(f"Total combinations to test: {total_combinations}\n")

for i, params in enumerate(hyperparam_combinations):
    current_params = base_parameters.copy()
    current_params.update(params)
    
    print(f"--- Testing Combination {i+1}/{total_combinations}: {current_params} ---")

    # 1. Instantiate ReRaLSTM with the current set of hyperparameters
    temp_rera_lstm = ReRaLSTM(
        data_path=data_path,
        market_name=market_name,
        tickers_fname=tickers_fname,
        relation_name=relation_name,
        emb_fname=emb_fname,
        parameters=current_params,
        steps=1,
        epochs=50, # A consistent number of epochs for fair comparison
        batch_size=None,
        flat=params['flat'],
        gpu=torch.cuda.is_available(),
        in_pro=params['in_pro']
    )

    # 2. Train the model and get the performance of the best validation epoch
    _, _, _, _, test_pred, test_gt, test_mask, test_perf = temp_rera_lstm.train()
    
    # 3. Evaluate the performance, focusing on IRR (btl)
    current_irr = test_perf.get('btl', -float('inf'))
    
    print(f"Combination {i+1} Result: IRR (btl) = {current_irr:.4f}, MSE = {test_perf['mse']:.6f}, MRR = {test_perf['mrrt']:.4f}")
    
    # 4. Store and track the best results
    # Add a column to distinguish between RSR_E and RSR_I for clarity
    run_details = current_params.copy()
    run_details['model_type'] = 'RSR_E (Explicit)' if params['in_pro'] else 'RSR_I (Implicit)'
    results.append({**run_details, **test_perf})
    
    if current_irr > best_irr:
        best_irr = current_irr
        best_params = run_details
        best_perf = test_perf
        print(f"\n*** New Best IRR Found: {best_irr:.4f} ***\n")

# --- Display Final Results ---
print("\n" + "="*60)
print("Grid Search Complete!")
print("="*60)

results_df = pd.DataFrame(results)
print("Full Grid Search Results:")
print(results_df[['model_type', 'alpha', 'btl', 'mse', 'mrrt']].sort_values(by='btl', ascending=False).to_string())

print("\n" + "="*60)
print("Best Hyperparameters Found")
print("="*60)
print(f"Parameters: {best_params}")
print(f"Achieved IRR (btl): {best_perf.get('btl', 'N/A'):.4f}")
print(f"Achieved MSE: {best_perf.get('mse', 'N/A'):.6f}")
print(f"Achieved MRR: {best_perf.get('mrrt', 'N/A'):.4f}")